# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

Ce notebook transforme votre environnement **Kaggle GPU gratuit (30h/semaine)** en serveur pour votre studio **AbiArt**.

### ⚡ Configuration requise dans Kaggle (panneau à droite) :
1. **Accelerator** : Choisir `GPU T4 x2` (ou `P100`)
2. **Internet** : Activer `Internet ON`
3. Cliquez sur **Run All** (ou exécutez les cellules dans l'ordre).
4. À la dernière cellule, copiez l'URL générée (`https://xxxx.trycloudflare.com`) et collez-la dans AbiArt !

In [ ]:
# 1. Vérification du GPU
!nvidia-smi

In [ ]:
# 2. Résolution des dépendances (CLIP / setuptools / open_clip / Pillow)
!pip install -q "setuptools<70" wheel ftfy regex tqdm "Pillow>=10.0.0"
!pip install -q --no-build-isolation https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip
!pip install -q --no-build-isolation https://github.com/mlfoundations/open_clip/archive/bb44a442de3a22444740f64bba978707a0068305.zip
!pip install -q xformers ultralytics mediapipe pyngrok

print("✓ Dépendances de base installées !")

In [ ]:
# 3. Téléchargement de SD WebUI et pré-installation visible des requirements
# (L'installation directe évite le blocage silencieux « Installing requirements » de launch.py)
import os
%cd /kaggle/working

if not os.path.exists('/kaggle/working/stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /kaggle/working/stable-diffusion-webui

%cd /kaggle/working/stable-diffusion-webui

# Patch Pillow et dépôt Stability
os.environ["STABLE_DIFFUSION_REPO"] = "https://github.com/w-e-w/stablediffusion.git"
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements_versions.txt 2>/dev/null || true
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements.txt 2>/dev/null || true

# Installation directe avec progression visible
print("Installation des dépendances d'Automatic1111 (prend environ 60 secondes)...")
!pip install -r requirements_versions.txt --prefer-binary

# Pré-clonage des sous-dépôts requis dans repositories/
print("Préparation des sous-dépôts...")
!mkdir -p repositories
!git clone https://github.com/w-e-w/stablediffusion.git repositories/stable-diffusion-stability-ai 2>/dev/null || true
!git clone https://github.com/CompVis/taming-transformers.git repositories/taming-transformers 2>/dev/null || true
!git clone https://github.com/crowsonkb/k-diffusion.git repositories/k-diffusion 2>/dev/null || true
!git clone https://github.com/sczhou/CodeFormer.git repositories/CodeFormer 2>/dev/null || true
!git clone https://github.com/salesforce/BLIP.git repositories/BLIP 2>/dev/null || true

# Extensions (ADetailer, ControlNet)
!git clone --depth 1 https://github.com/Bing-su/adetailer.git extensions/adetailer 2>/dev/null || true
!git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git extensions/sd-webui-controlnet 2>/dev/null || true

print("✓ Tout est prêt !")

In [ ]:
# 4. Téléchargement de modèles par défaut (Animagine XL 3.1 & SDXL VAE)
%cd /kaggle/working/stable-diffusion-webui/models/Stable-diffusion

if not os.path.exists('animagine-xl-3.1.safetensors'):
    print("Téléchargement d'Animagine XL 3.1...")
    !wget -q -c https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -O animagine-xl-3.1.safetensors

%cd /kaggle/working/stable-diffusion-webui/models/VAE
if not os.path.exists('sdxl_vae.safetensors'):
    !wget -q -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -O sdxl_vae.safetensors

%cd /kaggle/working/stable-diffusion-webui
print("✓ Modèles prêts !")

In [ ]:
# 5. Installation de Cloudflare Tunnel (Tunnel gratuit sans compte)
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
print("✓ Cloudflared prêt !")

In [ ]:
# 6. Démarrage du Tunnel et de l'API Automatic1111 (Démarrage immédiat sans bloquer)
import os
import subprocess
import time
import re
import threading

os.environ["STABLE_DIFFUSION_REPO"] = "https://github.com/w-e-w/stablediffusion.git"

# Démarrer le tunnel Cloudflare en tâche de fond
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

def print_tunnel_url():
    for line in iter(tunnel_proc.stderr.readline, ''):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🎨 URL KAGGLE POUR ABIART :")
            print(f"   >>>  {url}  <<<")
            print("="*60)
            print("Copiez cette URL et collez-la dans AbiArt → Backend: Kaggle → Connect\n")
            break

threading.Thread(target=print_tunnel_url, daemon=True).start()
time.sleep(2)

# Lancement immédiat avec --skip-install (car les dépendances sont déjà prêtes)
!python launch.py \
    --skip-install \
    --skip-torch-cuda-test \
    --enable-insecure-extension-access \
    --no-half-vae \
    --xformers \
    --api \
    --listen \
    --port 7860